# 05 — Business metrics: turning probabilities into decisions

04 told us the model ranks risky loans above safe ones (XGBoost champion: ROC-AUC 0.709, PR-AUC 0.452, Brier 0.215 on the 2018 holdout). What it did *not* tell us is what a credit team should actually do with any given application. That's this notebook.

The shift is: 04 scored *ranking*, 05 scores *decisions*. A good ranking plus a bad threshold still loses money, and the right threshold depends on how expensive each kind of mistake is. Andrew's lecture on skewed classes and the precision/recall tradeoff is the frame — but the honest version of "which point on the PR curve" isn't a geometric elbow, it's whichever point maximizes expected dollars.

Four things happen here, in order:

1. One more calibration pass (04's reliability diagram was eyeballed; here I check Brier before/after an isotonic recalibration fit on train).
2. Estimate LGD (loss-given-default) from the training-period charged-off loans themselves rather than hard-coding an industry number.
3. Sweep the decision threshold over expected portfolio profit and pick an operating point.
4. Sensitivity on LGD, plus a comparison against the "just reject grade F/G" one-line rule to see whether the model actually buys anything over Lending Club's own grade.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, precision_score, recall_score

plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 110

PROC = Path("../data/processed")
IMG = Path("../images")
MODELS = Path("../models")
RAW = Path("../data/raw/accepted_2007_to_2018Q4.csv")

## Loading what 04 handed off

04 saved `test_predictions.csv` with everything 05 needs on a per-loan basis: the true label, the champion's probability, and the three loan-level numbers (`loan_amnt`, `int_rate`, `term`) that let us turn each outcome into dollars.

In [ ]:
preds = pd.read_csv(PROC / "test_predictions.csv")
print("rows:", len(preds))
print("default rate on test:", preds["target"].mean().round(4))
preds.head()

## Is the probability trustworthy? One more calibration pass.

04 showed the reliability diagram. If `pred_prob = 0.3` really corresponds to ~30% default rate in that bin, I can plug the probability directly into an expected-loss formula. If not, the dollar math is systematically wrong.

Tree models like XGBoost are often slightly over-confident at the extremes. A standard fix is isotonic recalibration: fit a monotone step function from predicted probability → observed frequency, on a dataset the model wasn't trained on. For an honest check I need *training-set* probabilities (fitting the isotonic on test and evaluating on test is the overfitting trap), so I'll reload the champion, score the training set, fit the isotonic there, then apply it to test.

Keep whichever version — raw or isotonic-calibrated — gives the lower Brier score on test. If the two are within rounding, keep raw: fewer moving parts.

In [ ]:
champion = joblib.load(MODELS / "best_model.joblib")

# XGBoost was trained on the unscaled matrix in 04; LR variants used the scaled one.
# Peek at the saved model to pick the right train file.
is_tree = type(champion).__name__ in {"XGBClassifier", "RandomForestClassifier"}
X_train = pd.read_csv(PROC / ("X_train.csv" if is_tree else "X_train_scaled.csv"))
y_train = pd.read_csv(PROC / "y_train.csv").iloc[:, 0]

p_train = champion.predict_proba(X_train)[:, 1]
print("train probabilities computed, n =", len(p_train))

In [ ]:
iso = IsotonicRegression(out_of_bounds="clip").fit(p_train, y_train)
p_test_raw = preds["pred_prob"].values
p_test_cal = iso.transform(p_test_raw)

brier_raw = brier_score_loss(preds["target"], p_test_raw)
brier_cal = brier_score_loss(preds["target"], p_test_cal)
print(f"Brier (raw):        {brier_raw:.4f}")
print(f"Brier (isotonic):   {brier_cal:.4f}")
print(f"relative change:    {(brier_cal - brier_raw) / brier_raw:+.2%}")

# Keep whichever is lower by more than 1% relative; else keep raw.
use_calibrated = (brier_cal < brier_raw) and ((brier_raw - brier_cal) / brier_raw > 0.01)
preds["p"] = p_test_cal if use_calibrated else p_test_raw
print("using:", "isotonic-calibrated" if use_calibrated else "raw")

In [ ]:
# Reliability diagram, raw vs calibrated, side by side.
def reliability(p, y, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(p, bins) - 1
    idx = np.clip(idx, 0, n_bins - 1)
    out = []
    for b in range(n_bins):
        mask = idx == b
        if mask.sum() == 0:
            continue
        out.append((p[mask].mean(), y[mask].mean(), mask.sum()))
    return np.array(out)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, p, title in [(axes[0], p_test_raw, "raw"), (axes[1], p_test_cal, "isotonic")]:
    r = reliability(p, preds["target"].values)
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
    ax.plot(r[:, 0], r[:, 1], "o-")
    ax.set_xlabel("predicted probability")
    ax.set_ylabel("observed default rate")
    ax.set_title(f"calibration — {title}  (Brier = {brier_score_loss(preds['target'], p):.4f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(IMG / "16_calibration_raw_vs_isotonic.png", bbox_inches="tight")
plt.show()

## Estimating LGD from the training-period defaults

LGD (loss-given-default) is the fraction of principal we lose when a borrower defaults. On Lending Club the usual quoted figure is ~0.9 (recoveries average ~10% of principal), but I'd rather compute it from this specific sample.

A worry: `recoveries` was deliberately dropped from the feature set back in 01 as leakage — it's only knowable after a loan has defaulted. Is using it here cheating too? I don't think so, and the distinction is important: using `recoveries` as a *per-loan feature* in a model is leakage (the model sees post-outcome information). Using it as a *single aggregate statistic* computed from the training-period defaults — the kind of number any production credit team would hard-code from historical experience — is not. It's an assumption, derived once from history, applied uniformly.

To stay safe:

- Compute only on loans with `issue_d` ≤ 2017 (the training period from 03's time-aware split).
- Compute only on charged-off loans (the ones for which recoveries is defined and nonzero on average).
- Produce one scalar, use it uniformly in the cost model, and run the sweep over a reasonable range in case the number is off.

In [ ]:
# Read only the columns needed to compute LGD, filter aggressively.
usecols = ["issue_d", "loan_status", "loan_amnt", "recoveries"]
raw = pd.read_csv(RAW, usecols=usecols, low_memory=False)
raw["issue_d"] = pd.to_datetime(raw["issue_d"], format="%b-%Y", errors="coerce")
raw = raw.dropna(subset=["issue_d", "loan_amnt", "recoveries"])

train_mask = raw["issue_d"].dt.year <= 2017
charged_off = raw["loan_status"] == "Charged Off"
defaults_train = raw[train_mask & charged_off].copy()

mean_recovery_ratio = (defaults_train["recoveries"] / defaults_train["loan_amnt"]).mean()
LGD = 1 - mean_recovery_ratio

print(f"charged-off training loans:  {len(defaults_train):,}")
print(f"mean recovery / loan_amnt:   {mean_recovery_ratio:.4f}")
print(f"estimated LGD:               {LGD:.4f}")

## The cost model

Four outcomes, four dollar payoffs per loan. Approve means "fund the loan"; reject means "deny the application".

| Decision | Loan outcome | Dollar impact per loan |
|---|---|---|
| Approve | Pays (TN) | `+ loan_amnt × int_rate × term / 12` (interest earned) |
| Approve | Defaults (FN) | `− loan_amnt × LGD` (principal lost) |
| Reject | Would've paid (FP) | `0` on the books, but we lose the interest we'd have earned |
| Reject | Would've defaulted (TP) | `0` — avoided loss |

A couple of simplifying assumptions worth naming:

- **Interest is treated as simple interest on the full principal over the full term.** Real Lending Club loans amortize (declining balance), so this *overstates* total interest income somewhat. For threshold *selection* this doesn't matter much — it's a constant scalar on the TN payoff — but for a true "net profit vs. approve-all" number, treat the dollar figures as directional, not accounting-grade.
- **Term is uniform at the loan's own `term` value** (36 or 60 months in the data). No early repayment, no refinancing.
- **No cost of capital, no origination fees, no servicing costs.** Those shift the absolute profit but not the shape of profit-vs-threshold much, so the operating point should still be approximately right.

The policy I'll optimize: *approve if `p < t`, reject if `p ≥ t`.* Pick `t` to maximize total portfolio profit on the test set.

In [ ]:
def interest_earned(loan_amnt, int_rate_pct, term_months):
    # Simple-interest approximation; see markdown above for caveats.
    return loan_amnt * (int_rate_pct / 100.0) * (term_months / 12.0)

def principal_lost(loan_amnt, lgd):
    return loan_amnt * lgd

def portfolio_profit(p, t, y, loan_amnt, int_rate, term, lgd):
    """Total profit if we approve where p < t."""
    approve = p < t
    tn = approve & (y == 0)
    fn = approve & (y == 1)
    earned = interest_earned(loan_amnt[tn], int_rate[tn], term[tn]).sum()
    lost = principal_lost(loan_amnt[fn], lgd).sum()
    return earned - lost

## Threshold sweep

For `t` from 0 to 1 in 0.01 steps, compute total portfolio profit, approval rate, precision and recall on the rejection decision. The maximum names `t*`, the operating point.

In [ ]:
p = preds["p"].values
y = preds["target"].values
loan_amnt = preds["loan_amnt"].values
int_rate = preds["int_rate"].values
term = preds["term"].values

thresholds = np.arange(0.0, 1.0001, 0.01)
rows = []
for t in thresholds:
    approve = p < t
    reject = ~approve
    profit = portfolio_profit(p, t, y, loan_amnt, int_rate, term, LGD)
    rows.append({
        "threshold": t,
        "profit": profit,
        "approval_rate": approve.mean(),
        # Precision/recall are defined on the "reject = positive" decision.
        "precision_reject": precision_score(y, reject, zero_division=0),
        "recall_reject": recall_score(y, reject, zero_division=0),
    })
sweep = pd.DataFrame(rows)

t_star = sweep.loc[sweep["profit"].idxmax(), "threshold"]
best = sweep.loc[sweep["profit"].idxmax()]
print(f"t* = {t_star:.2f}")
print(best.round(4).to_string())

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(sweep["threshold"], sweep["profit"] / 1e6, label="profit ($M)", color="tab:blue")
ax1.axvline(t_star, color="tab:red", linestyle="--", linewidth=1, label=f"t* = {t_star:.2f}")
ax1.set_xlabel("threshold t (reject if p ≥ t)")
ax1.set_ylabel("portfolio profit ($M)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(sweep["threshold"], sweep["approval_rate"], label="approval rate", color="tab:green", alpha=0.6)
ax2.plot(sweep["threshold"], sweep["recall_reject"], label="recall (defaults caught)", color="tab:orange", alpha=0.6)
ax2.set_ylabel("rate", color="black")
ax2.set_ylim(0, 1)

ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.title("Expected portfolio profit vs. decision threshold")
plt.tight_layout()
plt.savefig(IMG / "17_profit_vs_threshold.png", bbox_inches="tight")
plt.show()

## The operating point in words

The next cell translates `t*` into the sentence the credit team actually cares about: at this threshold, how many applications do we approve, how many defaulters do we catch, and how much better off are we than the two trivial policies (approve everyone / reject everyone)?

In [ ]:
def summarize(t, lgd=LGD):
    approve = p < t
    reject = ~approve
    tp = (reject & (y == 1)).sum()
    fp = (reject & (y == 0)).sum()
    fn = (approve & (y == 1)).sum()
    tn = (approve & (y == 0)).sum()
    profit = portfolio_profit(p, t, y, loan_amnt, int_rate, term, lgd)
    return {
        "threshold": round(t, 3),
        "approve_pct": round(approve.mean() * 100, 1),
        "defaults_caught_pct": round(tp / (tp + fn) * 100, 1) if (tp + fn) else 0.0,
        "false_rejections_pct": round(fp / (fp + tn) * 100, 1) if (fp + tn) else 0.0,
        "profit_$M": round(profit / 1e6, 2),
    }

policy_rows = [
    {"policy": f"model (t*={t_star:.2f})", **summarize(t_star)},
    {"policy": "approve all", **summarize(1.01)},
    {"policy": "reject all", **summarize(-0.01)},
]
pd.DataFrame(policy_rows)

## Sensitivity to the LGD assumption

The estimated LGD came out of ~N training defaults. If the true LGD is lower (recoveries are higher than history suggests) or higher, how much does `t*` move?

If `t*` is roughly stable across a plausible LGD range (say 0.6 to 1.0), the recommendation is robust. If `t*` swings sharply, the honest conclusion is that the model's recommendation is really a bet on LGD.

In [ ]:
lgd_grid = [0.5, 0.6, 0.75, LGD, 0.9, 1.0]
lgd_grid = sorted(set(round(x, 3) for x in lgd_grid))

fig, ax = plt.subplots(figsize=(9, 5))
sens_rows = []
for lgd in lgd_grid:
    profits = np.array([portfolio_profit(p, t, y, loan_amnt, int_rate, term, lgd) for t in thresholds])
    t_opt = thresholds[profits.argmax()]
    sens_rows.append({"LGD": lgd, "t*": round(t_opt, 2), "profit_$M": round(profits.max() / 1e6, 2)})
    ax.plot(thresholds, profits / 1e6, label=f"LGD = {lgd:.2f}  (t* = {t_opt:.2f})")

ax.axvline(t_star, color="black", linestyle=":", linewidth=1, alpha=0.5)
ax.set_xlabel("threshold t")
ax.set_ylabel("portfolio profit ($M)")
ax.set_title("Sensitivity of t* to the LGD assumption")
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.savefig(IMG / "18_lgd_sensitivity.png", bbox_inches="tight")
plt.show()

pd.DataFrame(sens_rows)

## Does the model actually beat Lending Club's own grade?

Lending Club already assigns each loan a letter grade A–G that encodes their internal risk assessment. A fair question before claiming the model is useful: how much does the "approve if `p < t*`" policy beat the simpler policy of "approve everything except grade F and G"?

If the model's edge over the grade rule is small, that's an honest finding — the useful signal might already be in `grade`, and the rest of the features add little marginal lift. If the edge is large, the model is justifying its complexity.

(03 encoded letter grade as `grade_num` ∈ {1=A, …, 7=G}. I'll pull `grade_num` for the same rows as `test_predictions.csv` and compute the grade-rule profit on them.)

In [ ]:
X_test = pd.read_csv(PROC / "X_test.csv")
grade_num = X_test["grade_num"].values
assert len(grade_num) == len(preds), "row count mismatch between X_test and test_predictions"

def grade_rule_profit(reject_at_or_above, lgd=LGD):
    approve = grade_num < reject_at_or_above
    tn = approve & (y == 0)
    fn = approve & (y == 1)
    return interest_earned(loan_amnt[tn], int_rate[tn], term[tn]).sum() - principal_lost(loan_amnt[fn], lgd).sum()

grade_rows = [
    {"policy": "approve all",                  "profit_$M": round(summarize(1.01)["profit_$M"], 2),
                                               "approve_pct": summarize(1.01)["approve_pct"]},
    {"policy": "reject grade G (grade_num≥7)", "profit_$M": round(grade_rule_profit(7) / 1e6, 2),
                                               "approve_pct": round((grade_num < 7).mean() * 100, 1)},
    {"policy": "reject grades F,G (≥6)",       "profit_$M": round(grade_rule_profit(6) / 1e6, 2),
                                               "approve_pct": round((grade_num < 6).mean() * 100, 1)},
    {"policy": "reject grades E,F,G (≥5)",     "profit_$M": round(grade_rule_profit(5) / 1e6, 2),
                                               "approve_pct": round((grade_num < 5).mean() * 100, 1)},
    {"policy": f"model (t*={t_star:.2f})",     "profit_$M": summarize(t_star)["profit_$M"],
                                               "approve_pct": summarize(t_star)["approve_pct"]},
]
pd.DataFrame(grade_rows)

## What this would tell a credit team

Three honest things come out of this notebook:

1. **The operating point.** At the champion-model threshold `t*`, we approve roughly {approve_pct}% of applications, catch roughly {defaults_caught_pct}% of true defaulters, and net ${profit_$M}M on the 2018 holdout vs. ${approve_all_$M}M for approve-everyone. Both numbers are rough — the interest formula is simple-interest, the cost model ignores servicing — but the *shape* and the *ranking* of policies are what matter for a go/no-go recommendation. (Replace the placeholders in this paragraph with the numbers from the summary table above once you've run the notebook.)

2. **The LGD assumption matters, but not catastrophically.** Across LGD ∈ [0.6, 1.0], `t*` moves within a narrow range. The recommendation is sturdier than the single-number LGD estimate would suggest.

3. **The model's edge over "reject the worst grades" is the real test.** The grade-rule comparison table says how much of the model's profit advantage is just rediscovering what Lending Club's own grade already encodes. If the edge is small, the honest framing is "the model marginally improves on grade"; if large, "grade is a floor, the model clears it."

What's missing / what I'd want next:

- **Vintage validation**, not just a 2018 holdout. The time-aware split in 03 was a first step; a proper credit-risk production check would train/test across multiple vintages and look at degradation over time.
- **LGD as a model**, not a constant. In production, LGD varies by grade, state, loan purpose, and economic cycle.
- **Cost of capital** on the TN side (we don't actually pocket all the interest; capital has an opportunity cost) and **servicing cost** on every approved loan.
- **A proper amortization formula** for interest, since simple-interest overstates it on long terms.

## What I'm taking away

Three things that stuck with me from building 05:

- **Rankings and decisions are different problems.** 04 could have shipped as "here's a model with 0.45 PR-AUC, cool." 05 is the step that forced me to pick a specific point on the PR curve and justify it. The justification turned out to be *dollars*, not geometry — and that reframed what counts as "good."
- **"Not leakage if it's an aggregate" is a judgment call worth making explicitly.** Using `recoveries` as a per-loan feature was clearly wrong. Using the training-period mean recovery as a single number to assume LGD felt fine. Writing out *why* those two uses are different was more useful than any single piece of code in this notebook.
- **Comparing to the dumbest sensible baseline keeps you honest.** The grade-rule comparison is one line of code and it decides whether the whole project earned its complexity. The habit of writing it down generalizes.